# Eva Data

This notebook builds a `MultiDataset` containing exactly one `ZarrDataset`, loads one batch, visualizes one image with `mediapy`, and prints the rest of the batch.

In [ ]:
from pathlib import Path

import cv2
import imageio_ffmpeg
import mediapy as mpy
import numpy as np
import torch

from egomimic.rldb.embodiment.eva import Eva
from egomimic.rldb.embodiment.human import Aria
from egomimic.rldb.zarr.zarr_dataset_multi import MultiDataset, ZarrDataset
from egomimic.rldb.zarr.zarr_dataset_multi import S3EpisodeResolver
from egomimic.utils.egomimicUtils import (
    INTRINSICS,
    cam_frame_to_cam_pixels,
    nds,
)
from egomimic.utils.aws.aws_data_utils import load_env

# Ensure mediapy can find an ffmpeg executable in this environment
mpy.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

In [ ]:
TEMP_DIR = "/tmp/" # replace with your own temp directory for caching S3 data
load_env()

In [ ]:
key_map = Eva.get_keymap()
transform_list = Eva.get_transform_list()

resolver = S3EpisodeResolver(
    TEMP_DIR, key_map=key_map, transform_list=transform_list
)
filters = {
    "episode_hash": "2026-01-03-02-16-02-519000"
}
multi_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader = torch.utils.data.DataLoader(multi_ds, batch_size=1, shuffle=False)

In [ ]:
# Separate YPR visualization preview
for batch in loader:
    vis_ypr = Eva.viz_transformed_batch(batch, mode="palm_axes")
    mpy.show_image(vis_ypr)
    break

In [ ]:
images = []
for i, batch in enumerate(loader):
    vis = Eva.viz_transformed_batch(batch, mode="palm_traj")
    images.append(vis)
    if i > 10:
        break

mpy.show_video(images, fps=30)

## Human Datasets
Mecka, Scale and Aria should all run exactly the same

In [ ]:
intrinsics_key = "base"

key_map = Aria.get_keymap()
transform_list = Aria.get_transform_list()

resolver = S3EpisodeResolver(
    TEMP_DIR,
    key_map=key_map,
    transform_list=transform_list,
)

filters = {"episode_hash": "2025-11-27-23-14-02-277000"} #aria
# filters = {"episode_hash": "692ee048ef7557106e6c4b8d"} # mecka

cloudflare_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader = torch.utils.data.DataLoader(cloudflare_ds, batch_size=1, shuffle=False)

In [ ]:
ims = []
for i, batch in enumerate(loader):
    vis = Aria.viz_transformed_batch(batch, mode="palm_traj")
    ims.append(vis)
    # mpy.show_image(vis)

    # for k, v in batch.items():
    #     print(f"{k}: {tuple(v.shape)}")
    
    if i > 10:
        break

mpy.show_video(ims, fps=30)


In [ ]:
# Aria YPR video (same data loop, YPR overlay)
ims_ypr = []
for i, batch in enumerate(loader):
    vis_ypr = Aria.viz_transformed_batch(batch, mode="palm_axes")
    ims_ypr.append(vis_ypr)
    if i > 20:
        break

mpy.show_video(ims_ypr, fps=30)

## Aria Gaze

In [ ]:
from egomimic.utils.egomimicUtils import INTRINSICS, cam_frame_to_cam_pixels, draw_dot_on_frame, get_gaze_endpoint, ARIA_T_RGB_CPF
from egomimic.rldb.zarr.zarr_dataset_multi import MultiDataset, ZarrDataset
import mediapy as mpy
import numpy as np
import torch
from pathlib import Path

key_map_gaze = {
    "images.front_1": {"zarr_key": "images.front_1"},
    "eye_gaze": {"zarr_key": "obs_eye_gaze"},
}
intrinsics = INTRINSICS["base"]


filters = {"episode_hash": "2025-11-27-23-14-02-277000"} #aria
# filters = {"episode_hash": "692ee048ef7557106e6c4b8d"} # mecka
resolver = S3EpisodeResolver(
    TEMP_DIR, key_map=key_map_gaze
)
cloudflare_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader = torch.utils.data.DataLoader(cloudflare_ds, batch_size=1, shuffle=False)

In [ ]:
ims_gaze = []
for i, batch in enumerate(loader):
    print(batch.keys())
    gaze_data = batch['eye_gaze'][0].numpy() # yaw (rads), pitch (rads), depth (m)
    if gaze_data[0] != -100: # value for no gaze estimate
        gaze_point_xyz = get_gaze_endpoint(gaze_data[0], gaze_data[1], gaze_data[2], ARIA_T_RGB_CPF)
        gaze_point_xyz = np.expand_dims(gaze_point_xyz, 0)
        gaze_point_pixel = cam_frame_to_cam_pixels(gaze_point_xyz, intrinsics)
        frame = batch['images.front_1'][0].permute(1,2,0).numpy() * 255
        vis = draw_dot_on_frame(frame, gaze_point_pixel, show=False)
        ims_gaze.append(vis)
        if i > 600:
            break

In [ ]:
mpy.show_video(ims_gaze, fps=30)